In [1]:
import os
import mlflow

os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"

mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")

In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/ced16f5d8bce44b5a9c3e335b0f484b4', creation_time=1784571024655, experiment_id='9', last_update_time=1784571024655, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import ADASYN
import mlflow
import mlflow.sklearn
import optuna

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
!pip install optuna

  Using cached optuna-4.9.0-py3-none-any.whl.metadata (15 kB)
Using cached optuna-4.9.0-py3-none-any.whl (425 kB)

   ---------------------------------------- 0/2 [colorlog]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ----


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Dell\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
df = pd.read_csv('dataset.csv').dropna()
df.shape

(36662, 2)

In [8]:
# Step 1: Load and clean data
#df = pd.read_csv('dataset.csv').dropna()
df = df.dropna(subset=['category'])

# Step 2: Split FIRST on raw text to avoid data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Step 3: TF-IDF vectorizer - FIT ONLY on training data
ngram_range = (1, 2)  # Bigram
max_features = 10000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train = vectorizer.fit_transform(X_train_raw)  # Fit ONLY on training data
X_test = vectorizer.transform(X_test_raw)  # Transform test data (don't refit)

# Step 4: Apply ADASYN ONLY to training data to handle class imbalance
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)

# Test data stays untouched for fair evaluation
X_train = X_train_resampled
y_train = y_train_resampled

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type and experiment info
        mlflow.set_tag("mlflow.runName", f"{model_name}_ADASYN_TFIDF_Bigrams_NoLeakage")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.set_tag("data_handling", "no_leakage_fixed")
        mlflow.set_tag("model_type", "KNeighborsClassifier")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("imbalance_handling", "ADASYN_on_train_only")
        mlflow.log_param("n_neighbors", model.n_neighbors)
        mlflow.log_param("p", model.p)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        
        print(f"Run logged successfully for {model_name}")


# Step 5: Optuna objective function for KNN
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 3, 30)
    p = trial.suggest_categorical('p', [1, 2])

    model = KNeighborsClassifier(n_neighbors=n_neighbors, p=p)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


# Step 6: Run Optuna for KNN, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_knn, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = KNeighborsClassifier(n_neighbors=best_params['n_neighbors'], p=best_params['p'])

    # Log the best model with MLflow
    log_mlflow("KNN", best_model, X_train, X_test, y_train, y_test)
    
    print(f"\nBest Optuna parameters: {best_params}")
    print(f"Best trial accuracy: {study.best_value:.4f}")

# Run the experiment for KNN
run_optuna_experiment()

[I 2026-07-21 01:05:25,068] A new study created in memory with name: no-name-6d8b1dcb-4346-456e-9058-591a47899765
[I 2026-07-21 01:05:59,120] Trial 0 finished with value: 0.35074321560070915 and parameters: {'n_neighbors': 10, 'p': 2}. Best is trial 0 with value: 0.35074321560070915.
[I 2026-07-21 01:07:21,659] Trial 1 finished with value: 0.3540160916405291 and parameters: {'n_neighbors': 15, 'p': 1}. Best is trial 1 with value: 0.3540160916405291.
[I 2026-07-21 01:07:45,792] Trial 2 finished with value: 0.32756034365198416 and parameters: {'n_neighbors': 26, 'p': 2}. Best is trial 1 with value: 0.3540160916405291.
[I 2026-07-21 01:08:02,780] Trial 3 finished with value: 0.33683349243147415 and parameters: {'n_neighbors': 15, 'p': 2}. Best is trial 1 with value: 0.3540160916405291.
[I 2026-07-21 01:08:49,941] Trial 4 finished with value: 0.3497886267557616 and parameters: {'n_neighbors': 19, 'p': 1}. Best is trial 1 with value: 0.3540160916405291.
[I 2026-07-21 01:09:23,967] Trial 5 f

Run logged successfully for KNN

Best Optuna parameters: {'n_neighbors': 3, 'p': 2}
Best trial accuracy: 0.3985
